In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv
/kaggle/input/datasets/keyadobriyal/f1-strategy-dataset/f1_strategy_dataset_v4.csv


In [2]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import LabelEncoder

import optuna

# Models
from catboost import CatBoostClassifier, CatBoostError
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

class CFG:
    seed = 42
    target = "PitNextLap"
    id_col = "id"
    valid_size = 0.20
    
    # Global task type to easily toggle between CPU/GPU for the whole ensemble
    # Options: "GPU" or "CPU"
    device_type = "GPU" 

# Define functions for hyperparameter tuning
def tune_catboost(X_train, y_train, X_valid, y_valid, cat_features):
    def objective(trial):
        params = {
            "iterations": trial.suggest_int("iterations", 500, 5000, step=500),
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
            "depth": trial.suggest_int("depth", 4, 12),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
            # Fixed: Removed 'Poisson' which is GPU-only
            "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli"]),
            "eval_metric": "AUC",
            "random_seed": CFG.seed,
            "allow_writing_files": False,
            "verbose": 0
        }
        
        # Handle bootstrap-specific dependencies
        if params["bootstrap_type"] == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0.1, 1.0)
        elif params["bootstrap_type"] == "Bernoulli":
            # Bernoulli requires a subsample parameter between 0 and 1
            params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)

        model = CatBoostClassifier(**params)
        model.fit(X_train, y_train, eval_set=(X_valid, y_valid), cat_features=cat_features, early_stopping_rounds=300, verbose=False)
        preds = model.predict_proba(X_valid)[:, 1]
        return roc_auc_score(y_valid, preds)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)
    return study.best_params
    
def tune_xgboost(X_train, y_train, X_valid, y_valid):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 500, 5000, step=500),
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "tree_method": "hist",
            "early_stopping_rounds": 200,  # Fixed: Moved to initialization
            "eval_metric": "logloss",
            "random_state": CFG.seed,
            "verbosity": 0
        }
        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
        preds = model.predict_proba(X_valid)[:,1]
        return roc_auc_score(y_valid, preds)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)
    return study.best_params

def tune_lightgbm(X_train, y_train, X_valid, y_valid):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 500, 5000, step=500),
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 255),
            "max_depth": trial.suggest_int("max_depth", -1, 15),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 50), # Fixed: min_data_in_leaf alias
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
            "verbose": -1,
            "n_jobs": -1,
            "seed": CFG.seed
        }
        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
        preds = model.predict_proba(X_valid)[:, 1]
        return roc_auc_score(y_valid, preds)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)
    return study.best_params

# Auto-ensemble weight optimization
def optimize_ensemble_weights(preds_list, y_valid):
    def score(weights):
        ensemble_preds = np.zeros_like(y_valid, dtype=float)
        for weight, preds in zip(weights, preds_list):
            ensemble_preds += preds * weight
        return -roc_auc_score(y_valid, ensemble_preds)

    from scipy.optimize import minimize

    initial_weights = [1/len(preds_list)] * len(preds_list)
    constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)},)
    bounds = [(0, 1)] * len(preds_list)

    res = minimize(score, initial_weights, bounds=bounds, constraints=constraints)
    return res.x

def engineer_features(df):
    out = df.copy()
    eps = 1e-6
    # Strategy Logic
    if "LapNumber" in out.columns and "RaceProgress" in out.columns:
        out["EstTotalLaps"] = (out["LapNumber"] / (out["RaceProgress"] + eps)).clip(0, 100)
        out["LapsRemaining"] = out["EstTotalLaps"] - out["LapNumber"]
    # Tyre Wear
    if "TyreLife" in out.columns:
        out["TyreAgeRatio"] = out["TyreLife"] / (out["LapNumber"] + 1)
    return out
    
def run_pipeline():
    seed_everything(CFG.seed)
    
    # Update paths dynamically if needed, keeping your exact strings intact
    train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
    test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/test.csv")
    
    train = engineer_features(train)
    test = engineer_features(test)
    
    features = [c for c in train.columns if c not in [CFG.target, CFG.id_col]]
    cat_features = train[features].select_dtypes(include=['object', 'string']).columns.tolist()

    for col in cat_features:
        le = LabelEncoder()
        full_text = pd.concat([train[col], test[col]]).astype(str)
        le.fit(full_text)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))

    X = train[features]
    y = train[CFG.target].astype(int)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=CFG.valid_size, stratify=y, random_state=CFG.seed)

    # Tuning CatBoost
    print("Tuning CatBoost...")
    best_cat_params = tune_catboost(X_train, y_train, X_val, y_val, cat_features)
    print("Best CatBoost Params:", best_cat_params)
    
    # Fixed: Re-add static training configurations dropped by study.best_params
    best_cat_params.update({"eval_metric": "AUC", "allow_writing_files": False, "random_seed": CFG.seed})
    model_cb = CatBoostClassifier(**best_cat_params)
    model_cb.fit(X_train, y_train, eval_set=(X_val, y_val), cat_features=cat_features, verbose=500, early_stopping_rounds=300)

    # Tuning XGBoost
    print("Tuning XGBoost...")
    best_xgb_params = tune_xgboost(X_train, y_train, X_val, y_val)
    print("Best XGBoost Params:", best_xgb_params)
    
    # Fixed: Include initialization params
    best_xgb_params.update({"early_stopping_rounds": 200, "eval_metric": "logloss", "random_state": CFG.seed, "tree_method": "hist"})
    model_xgb = xgb.XGBClassifier(**best_xgb_params)
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    # Tuning LightGBM
    print("Tuning LightGBM...")
    best_lgb_params = tune_lightgbm(X_train, y_train, X_val, y_val)
    print("Best LightGBM Params:", best_lgb_params)
    
    best_lgb_params.update({"verbose": -1, "seed": CFG.seed, "n_jobs": -1})
    model_lgb = lgb.LGBMClassifier(**best_lgb_params)
    model_lgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])

    # Get validation predictions
    preds_cb = model_cb.predict_proba(X_val)[:, 1]
    preds_xgb = model_xgb.predict_proba(X_val)[:, 1]
    preds_lgb = model_lgb.predict_proba(X_val)[:, 1]
    preds_list = [preds_cb, preds_xgb, preds_lgb]

    # Optimize ensemble weights
    print("Optimizing ensemble weights...")
    weights = optimize_ensemble_weights(preds_list, y_val)
    print("Optimal Weights:", weights)

    # Final ensemble on validation
    final_val_preds = np.zeros_like(y_val, dtype=float)
    for weight, preds in zip(weights, preds_list):
        final_val_preds += preds * weight
    print(f"Final Ensemble AUC: {roc_auc_score(y_val, final_val_preds):.6f}")

    # Test predictions
    t1 = model_cb.predict_proba(test[features])[:, 1]
    t2 = model_xgb.predict_proba(test[features])[:, 1]
    t3 = model_lgb.predict_proba(test[features])[:, 1]
    final_test_preds = t1 * weights[0] + t2 * weights[1] + t3 * weights[2]

    submission = pd.DataFrame({CFG.id_col: test[CFG.id_col], CFG.target: final_test_preds})
    submission.to_csv("submission.csv", index=False)
    print("Submission Completed")
    print(submission.head(10))

# Run the pipeline
if __name__ == "__main__":
    run_pipeline()

[I 2026-05-18 00:27:39,911] A new study created in memory with name: no-name-dd9381a2-ac71-4e67-aca2-b1dfac7dacc5


Tuning CatBoost...


[I 2026-05-18 00:35:49,440] Trial 0 finished with value: 0.9287507219199163 and parameters: {'iterations': 3000, 'learning_rate': 0.0014639964038805537, 'depth': 4, 'l2_leaf_reg': 7.998263465218181, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.3793410334520593}. Best is trial 0 with value: 0.9287507219199163.
[I 2026-05-18 00:43:38,098] Trial 1 finished with value: 0.9506050645082353 and parameters: {'iterations': 1500, 'learning_rate': 0.02040152601350573, 'depth': 9, 'l2_leaf_reg': 1.2221153447414876, 'bootstrap_type': 'Bernoulli', 'subsample': 0.6984668816920492}. Best is trial 1 with value: 0.9506050645082353.
[I 2026-05-18 00:55:45,884] Trial 2 finished with value: 0.9483706752736643 and parameters: {'iterations': 4500, 'learning_rate': 0.027112607405303774, 'depth': 4, 'l2_leaf_reg': 7.086033799672924, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.26867923472898003}. Best is trial 1 with value: 0.9506050645082353.
[I 2026-05-18 01:05:09,962] Trial 3 finished wi

Best CatBoost Params: {'iterations': 2500, 'learning_rate': 0.052933800275980845, 'depth': 8, 'l2_leaf_reg': 2.685021202523274, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.12918518926272124}
0:	test: 0.9138067	best: 0.9138067 (0)	total: 315ms	remaining: 13m 7s
500:	test: 0.9493354	best: 0.9493354 (500)	total: 2m 29s	remaining: 9m 58s
1000:	test: 0.9509853	best: 0.9509859 (998)	total: 4m 57s	remaining: 7m 25s
1500:	test: 0.9514996	best: 0.9515007 (1499)	total: 7m 24s	remaining: 4m 55s
2000:	test: 0.9516778	best: 0.9516908 (1977)	total: 9m 53s	remaining: 2m 28s
2499:	test: 0.9517642	best: 0.9517784 (2352)	total: 12m 23s	remaining: 0us

bestTest = 0.9517783946
bestIteration = 2352

Shrink model to first 2353 iterations.


[I 2026-05-18 04:59:06,354] A new study created in memory with name: no-name-fa416dc4-64df-4d8c-9ff3-da5d1ecfd7b9


...

Best XGBoost Params: {'n_estimators': 5000, 'learning_rate': 0.004030886633352778, 'max_depth': 9, 'min_child_weight': 4, 'subsample': 0.8717249655044913, 'colsample_bytree': 0.6816749622928673}


[I 2026-05-18 05:33:26,152] A new study created in memory with name: no-name-6b8439dd-c28d-4897-b389-de72b3703d70


Tuning LightGBM...


[I 2026-05-18 05:35:02,298] Trial 0 finished with value: 0.9400719806162705 and parameters: {'n_estimators': 3000, 'learning_rate': 0.0012973042956900692, 'num_leaves': 37, 'max_depth': 7, 'min_child_samples': 36, 'feature_fraction': 0.7276865330701732, 'bagging_fraction': 0.7082888413415622, 'bagging_freq': 10}. Best is trial 0 with value: 0.9400719806162705.
[I 2026-05-18 05:35:24,986] Trial 1 finished with value: 0.8912680014575889 and parameters: {'n_estimators': 1500, 'learning_rate': 0.004461586958838065, 'num_leaves': 106, 'max_depth': 1, 'min_child_samples': 28, 'feature_fraction': 0.8223955713818879, 'bagging_fraction': 0.7778613221601607, 'bagging_freq': 3}. Best is trial 0 with value: 0.9400719806162705.
[I 2026-05-18 05:36:40,084] Trial 2 finished with value: 0.9513905629595243 and parameters: {'n_estimators': 3500, 'learning_rate': 0.017904710164422022, 'num_leaves': 231, 'max_depth': 13, 'min_child_samples': 42, 'feature_fraction': 0.9397126793588707, 'bagging_fraction': 

Best LightGBM Params: {'n_estimators': 2000, 'learning_rate': 0.025679376992182548, 'num_leaves': 169, 'max_depth': 15, 'min_child_samples': 31, 'feature_fraction': 0.6597084305745748, 'bagging_fraction': 0.6650452961100143, 'bagging_freq': 3}
Optimizing ensemble weights...
Optimal Weights: [0.33333333 0.33333333 0.33333333]
Final Ensemble AUC: 0.952585
Submission Completed
       id  PitNextLap
0  439140    0.004375
1  439141    0.005292
2  439142    0.003302
3  439143    0.140418
4  439144    0.864207
5  439145    0.233283
6  439146    0.001012
7  439147    0.003700
8  439148    0.016735
9  439149    0.001461
